# Material para as decisões do grupo

Campos `c5`, corpus `corp3`, tópicos `t5`. Quatro partes:
(a) aluguéis duvidosos, (b) os temas e seus nomes, (c) trocas de OCR da limpeza, (d) conferência de uma amostra.

Este caderno reúne o material para **quatro decisões do grupo**. Em cada parte há uma tabela com uma ou duas colunas em branco: preencha **fora do caderno** (numa cópia da tabela, planilha ou ata); o caderno só mostra os dados e não grava nada. Tudo é lido do banco na hora de executar, então os números mudam se o banco mudar.

## Preparação: ler o banco
O banco devolve no máximo 1000 linhas por consulta, então lemos em blocos (paginação). Este caderno **só lê**: não grava nada no banco nem em arquivos.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "scripts"))

import json, random
from IPython.display import HTML, display
import pandas as pd
from hdb import config, corpus
from hdb.banco import BancoSupabase
from resumo_dados import nomear   # nomes sugeridos dos temas (usa hdb.topicos.nomear quando existir)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)
banco = BancoSupabase()
db = banco.cliente

def ler_tudo(consulta, bloco=1000):
    """Executa a consulta em blocos de 1000 linhas e junta o resultado."""
    linhas, inicio = [], 0
    while True:
        parte = consulta().range(inicio, inicio + bloco - 1).execute().data
        linhas += parte
        if len(parte) < bloco:
            return pd.DataFrame(linhas)
        inicio += bloco

def periodo_curto(p):
    return "sem ano" if not isinstance(p, str) else (p if p.startswith("antes") else p.split(" ")[0])
PERIODOS = ["antes de 1831", "1831–1850", "1850–1871", "1871–1888", "sem ano"]

def trecho(texto, n):
    t = " ".join(str(texto).split())
    return t if len(t) <= n else t[:n].rstrip() + "…"

def mostrar(tabela, vazio):
    """Mostra a tabela e o n; se não há linhas, escreve a frase em vez de falhar."""
    if len(tabela) == 0:
        print(vazio)
        return
    print(f"n = {len(tabela)}")
    html = tabela.reset_index(drop=True).to_html(index=False, escape=True).replace("\n", "<br>")
    display(HTML("<style>.material td, .material th {text-align: left; vertical-align: top;}</style>"
                 + html.replace('class="dataframe"', 'class="dataframe material"')))

## Parte A. Aluguéis duvidosos
Anúncios de aluguel (`aluguel_incerto`) em que o texto não diz que a pessoa é escravizada.

In [2]:
inc = ler_tudo(lambda: db.table("anuncios_completos").select("id,periodo_legal,texto,c_evidencias")
               .in_("versao", list(config.VERSOES_ANUNCIOS)).eq("c_versao", config.VERSAO_CAMPOS)
               .eq("c_tipo", "aluguel_incerto").order("id"))
if len(inc) == 0:
    print("Não há anúncios de aluguel duvidoso nesta versão de campos.")
else:
    inc["periodo"] = pd.Categorical(inc["periodo_legal"].map(periodo_curto), categories=PERIODOS, ordered=True)
    print(f"{len(inc)} anúncios de aluguel duvidoso (campos {config.VERSAO_CAMPOS}).")
    if len(inc) != 189:
        print(f"ATENÇÃO: os comentários deste caderno foram escritos para 189 anúncios e agora são {len(inc)}: "
              "confira os números dos textos antes de usar.")
    por_periodo = inc.groupby("periodo", observed=True).size().rename("Anúncios").to_frame()
    por_periodo.index.name = "Período"
    print("\nPor período legal (n por linha):")
    display(por_periodo)

189 anúncios de aluguel duvidoso (campos c5).

Por período legal (n por linha):


,Anúncios
Período,
antes de 1831,1
1831–1850,17
1850–1871,90
1871–1888,81


In [3]:
def motivo(ev):
    if isinstance(ev, str):
        try:
            ev = json.loads(ev)
        except ValueError:
            return ev
    return (ev or {}).get("tipo", "(sem motivo registrado)")

if len(inc):
    amostra = inc.sample(min(20, len(inc)), random_state=42).sort_values("id")
    ex = pd.DataFrame({"id": amostra["id"], "Período": amostra["periodo"].astype(str),
                       "Texto (até 220 caracteres)": amostra["texto"].map(lambda t: trecho(t, 220)),
                       "Motivo (evidência do campo)": amostra["c_evidencias"].map(motivo)})
    print("20 exemplos sorteados (semente 42):")
    mostrar(ex, "Sem exemplos.")

20 exemplos sorteados (semente 42):
n = 20


id,Período,Texto (até 220 caracteres),Motivo (evidência do campo)
10756,1871–1888,"RECISA-SE de uma criada Hi nos, casa de pequena traventa e 8 Francisco de Pa “de costureiras € uia maclinisia; né Ouvidor mn. 4. Li de ur pequeno de lã a 18 aunos de idade, para um açougue; na rua do Engeuho-Velho , ou",pessoa sem marca de escravizado: criada
10767,1871–1888,"RECISA-SE, ne ras dz Oliado nº 0, do use Udo P * ingomuadeira, de bos ccndusta livro À (osECISA-SE de hqs “oficiças de” Prstaltas os olficina, divijase é 135, cobrado. ds pra ; FUI, um menino de 12 a 14 anuos de freloese…",pessoa sem marca de escravizado: menino
10843,1871–1888,"PRECISA-SE. para casa de Pequena familia, de uma rapariga que lave é engommes para tratar, a rua da Re fandega n. 28, andar, ás rar ESEECISA-HE duma ando Jo lodo ps ECISA-SE de uma criada de meia idade, para É casa de po…",pessoa sem marca de escravizado: rapariga
11710,1871–1888,"LUGA-SE ama prets que lava, cosinhe 6 em é com erianças; ne rua de 8. Pedro m. 160. a",pessoa sem marca de escravizado: ama
12493,1871–1888,"LUGA-SE uma que lava, cozinha e engom “A LUCAS ama Para cine rua da Quitanda 2: 49, a a Misericordia n. 116.",pessoa sem marca de escravizado: ama
12737,1871–1888,"RECISA-SE de uma criada, que saiba engommar roupa de senhora, lavar e cozinhar ; Carioca m. Há, sobrado.",pessoa sem marca de escravizado: criada
13028,1871–1888,"RECISA-SE de uma criada que cozinhe, para casa P Josquim » 132. na rua do Espirito-Santo de pequena familia; ENDE! perita eogom- Videira de toda a roupa e o mais serviço de",pessoa sem marca de escravizado: criada
13676,1871–1888,"RECISA-SE de uma perfei. a engommadeira, escrava ou livre; na rua da Quitanda",pessoa sem marca de escravizado: escrava ou livre
16742,1871–1888,"AMA Dk LEITE Precisa-se de uma, que tenha. leite do 7 meses à um ano; trata-se na rua Direita n. 3.",pessoa sem marca de escravizado: ama
18459,1871–1888,"LUGA-SE uma mocinha, vi À a arranjos de de família, para ama s da ma do General Camara n. 7 É & deccas é engomma O-SE mucamas, na rua do Oavilavadeiras e cozinheiras ;",pessoa sem marca de escravizado: mucama


**O que decidir:** o que fazer com os anúncios de aluguel em que o texto **não traz marca de pessoa escravizada** (`aluguel_incerto`): "aluga-se uma criada", "uma rapariga" etc. Podem ser pessoas escravizadas, forras ou livres.

**As 3 opções**

- **(a) Deixar fora** do corpus e dos tópicos (é o que fazemos hoje, variante `t5`).
- **(b) Incluir com marca**: entram no corpus e nas análises, sempre sinalizados como "incertos" (variante `t5i`).
- **(c) Incluir só com pista**: entram apenas os que têm alguma pista de condição (a evidência do campo diz qual).

**Recomendação: (b).** Já foi testada na variante `t5i`: **a direção do resultado não muda** (ver a seção de comparação no caderno `topicos_por_periodo`). Incluir com marca deixa o grupo ver os dois cenários sem perder esses anúncios.

## Parte B. Os temas
Para o modelo de tópicos da versão `t5`: nome sugerido, as 12 palavras de maior peso, quantos anúncios
têm aquele tema como o mais forte (n), de que tipos são e 5 anúncios típicos (os de maior proporção do tema).

In [4]:
MODELO = config.VERSAO_TOPICOS
pal = ler_tudo(lambda: db.table("topico_palavras").select("topico,palavra,peso").eq("versao", MODELO)
               .order("topico").order("peso", desc=True))
dist = ler_tudo(lambda: db.table("anuncio_topicos").select("anuncio_id,topico,proporcao").eq("versao", MODELO)
                .order("anuncio_id").order("topico"))
if len(pal) == 0 or len(dist) == 0:
    print(f"O modelo {MODELO} não tem dados no banco.")
else:
    ids = sorted(dist["anuncio_id"].unique())
    anu = ler_tudo(lambda: db.table("anuncios_completos").select("id,periodo_legal,c_tipo,texto")
                   .in_("versao", list(config.VERSOES_ANUNCIOS)).eq("c_versao", config.VERSAO_CAMPOS).order("id"))
    anu = anu[anu["id"].isin(ids)].drop_duplicates("id").set_index("id")
    palavras = {int(t): list(g["palavra"]) for t, g in pal.groupby("topico")}
    NOMES = nomear(palavras)
    pivo = dist.pivot(index="anuncio_id", columns="topico", values="proporcao").fillna(0)
    base = anu.join(pivo, how="inner")
    base["dominante"] = base[list(pivo.columns)].idxmax(axis=1)
    linhas = []
    for t in sorted(palavras):
        do_tema = base[base["dominante"] == t]
        tipos = do_tema["c_tipo"].value_counts(normalize=True)
        dist_tipo = ", ".join(f"{k} {100 * v:.0f}%" for k, v in tipos.items()) or "–"
        tip = base.sort_values(t, ascending=False).head(config.TOPICOS_TIPICOS)
        tipicos = "\n".join(f"• [id {i}, {r['c_tipo']}, {r[t]:.2f}] {trecho(r['texto'], 300)}" for i, r in tip.iterrows())
        linhas.append({"Tópico": f"T{t}", "Nome sugerido": NOMES[t], "12 palavras": ", ".join(palavras[t][:12]),
                       "n": len(do_tema), "Tipos (% dos anúncios do tema)": dist_tipo,
                       f"{config.TOPICOS_TIPICOS} anúncios típicos (300 caracteres)": tipicos,
                       "Nome escolhido pelo grupo": ""})
    print(f"{len(base)} anúncios no modelo {MODELO}; n = anúncios cujo tema mais forte é aquele.")
    mostrar(pd.DataFrame(linhas), "Sem temas.")

2167 anúncios no modelo t5; n = anúncios cujo tema mais forte é aquele.
n = 6


Tópico,Nome sugerido,12 palavras,n,Tipos (% dos anúncios do tema),5 anúncios típicos (300 caracteres),Nome escolhido pelo grupo
T0,Venda com preço e qualidades,"vende, preto, anos, bom, cozinheiro, aluga, moleque, precisa, servico, boa, idade, perfeito",577,"venda 61%, aluguel 34%, ama_de_leite 3%, compra 2%, fuga 1%","• [id 33669, venda, 0.98] VESDESE una cas é terreno, tendo este Gra E-SE um escravo, boa figura e moço, o qual ças de frente, sendo este proprio, com agua do padaria é é melio bom para ostro caniammento e tanque para lavar « Bicas correntes como pagem, porque entende de - em diversos lugares, aecommodações para fomilia ma na…\n• [id 38277, venda, 0.98] VENDE-SE hum preto de nação Caçange , idade de 18 annos, peritimo official de Pedreiro; na qua do Ouvidor n. 178. MUDANÇA DE CASA. TIHOMAZ Gosling aproveita esta occasião para agradecer ao respeitavel publico, a preferencia e protecção que recebeu durante a sua residencia na rua da Candelaria n. 6;…\n• [id 37793, venda, 0.98] 10 “ No largo do Paço n. 16, vendem-se varios escravos Jadinos) séndo hum perfeito cosinheiro dé tóda a qualidade de commida, e de 18 annos de idade, bom page, e comeprador; e outro de 17 annos, cosinheiro do ordinario, pers feito page, menta bem a cavall , e intende de Alfaiste , muito boa fizura e…\n• [id 10107, venda, 0.98] ENDEM-SE. na da Prainha m. 12%, ferra vos; um pardo perito enfermeiro, um moleque de annos, para roça, uma crioula de 20 anos, prendada, bos d do Suspiro n. ár loja, umapreta Mina quitandeira, bos peça. 2 telas 9 quartos, agua, quintal e tanque ro é toda forrada a papel; para tratar, no Tam VENDE SE…\n• [id 27966, venda, 0.97] VENDE-SE um escravo pardo, de boa prerens ça , de 26 a 28 annos de idade, perfeito official de sapateiro de toda a obra, perfeito boleciro de almofada e boléa, habil copeiro e pagem, sabendo tratar de animaes é refinar assucar, é é bom cozinheiro. Adverte-so que é a primeira vez que é vendido , e is…",
T1,Ama de leite,"aluga, leite, ama, preta, servico, bom, boa, meses, criancas, todo, carinhosa, parda",404,"ama_de_leite 47%, aluguel 41%, venda 10%, compra 1%, fuga 0%","• [id 13851, ama_de_leite, 0.97] AMAS DE LEITE 99 Quem quizer comprar huma escrava com muito leite, parida de 3 mezes, sabe lavar; à de barvelia, ensaboar, engomar, € COZ alguma consa; quem a pertender dirija-e a Tua da Mizericordia N. 45, que lá hade achar com quem tratar. 3) Quem quizer comprar huma preta com muito bora leite, e…\n• [id 38106, venda, 0.97] NENDE-SE húma: preta robusta, que sabe lavar | cer-na rua dós Pescadorés n. 1, ou dizer por este e cozinhar, com hum filho de 9 a 10 amnos de ida- 1 Jornal a suá tmorada, pata negocio do seu intevesse. de, e huma cria de 9 mezes; ua rua detraz plo Elosa PRECIS picio n. g5. A-SE alugar huma preta pat…\n• [id 13886, ama_de_leite, 0.97] o AMAS DE LEITE, a Quem tiver huma preta com leite, e cria, ara de criar huma criança, e que suioa laeozinhar , engomar, “e queira vender; pó- E y na rua do Cuno N. l44; adverteje procura tenha vicios. SD mA E q que nao of Quem quizer: alugar huma preta parida sem cria, com muito leite e bom; aia de…\n• [id 32756, ama_de_leite, 0.96] LUGAO-S! inheiro e copairo, por 308; uma co- AAum moleque, i nguezs; msc-atareira de rinhefra e lavadeira; uma. ; ums lavadeira e eogemmão e machica; ums Dº4 madeira, é uma ama, brancs, com bom leite ds um mez, com uma filha de 15 anncs, Aviso Esta encripiorio de ora em Emarem las à com) mesmas pras…\n• [id 34033, ama_de_leite, 0.96] ALUGA-SE. no beco de Bragança n.19, para :ma de leite, ou para qualquer cu ro serviço de portas a dentro, um: parda de 18 annos de idade, do puimeico parto, cu para quelquer serviço de portas a dentro — QUE preci-ar de uma ema com muito e bom leite, com todas as qualidade: para se lhe poder entregar…",
T2,Serviço doméstico (aluguel),"aluga, preta, lava, servico, cozinha, todo, engoma, lavar, faz, engomar, cozinhar, preto",545,"aluguel 74%, venda 23%, ama_de_leite 3%, compra 1%, fuga 0%","•

**O que decidir:** o nome de cada tema. Os nomes sugeridos vêm das palavras de cada tópico e são **hipóteses**. Leia as 12 palavras e os 5 anúncios típicos e preencha a coluna **Nome escolhido pelo grupo**. Lembre que os temas acompanham o tipo do anúncio (fuga, venda, aluguel, ama de leite): o nome pode seguir o tipo.

## Parte C. Trocas de OCR da limpeza
A limpeza descobre erros de OCR (palavra rara a uma letra de uma palavra muito mais frequente). Aqui repetimos a
descoberta sobre os anúncios selecionados do corpus, como em `corpus.processar`, **sem gravar nada**.

In [5]:
todos = banco.anuncios_para_corpus(config.VERSAO_CAMPOS)
sel = [a for a in todos if corpus.selecionar(a)]
if not sel:
    print("Nenhum anúncio selecionado para o corpus.")
else:
    padrao = [corpus.padronizar(a["texto"]) for a in sel]
    desc = corpus.descobrir_ocr(padrao)
    freq = {}
    for t in padrao:
        for p in set(t.split()):
            freq[p] = freq.get(p, 0) + 1
    print(f"{len(sel)} anúncios selecionados; {len(desc['aplicadas'])} trocas aplicadas, {len(desc['recusadas'])} recusadas.")
    ap = pd.DataFrame([{"Erro": w, "Correta": v, "Anúncios com o erro": freq.get(w, 0),
                        "Anúncios com a correta": freq.get(v, 0), "Aprovada? (s/n)": ""}
                       for w, v in desc["aplicadas"].items()]).sort_values("Anúncios com o erro", ascending=False) \
        if desc["aplicadas"] else pd.DataFrame()
    print("\nTrocas aplicadas:")
    mostrar(ap, "Nenhuma troca foi aplicada.")
    rec = pd.DataFrame([{"Palavra": w, "Candidata": v, "Anúncios com a palavra": n_w, "Anúncios com a candidata": n_v,
                         "Por que foi recusada": m} for w, v, n_w, n_v, m in desc["recusadas"]])
    if len(rec):
        rec = rec.sort_values("Anúncios com a palavra", ascending=False).head(10)
    print("\nAs 10 recusadas mais frequentes:")
    mostrar(rec, "Nenhuma troca foi recusada.")

2167 anúncios selecionados; 40 trocas aplicadas, 2696 recusadas.

Trocas aplicadas:
n = 40


Erro,Correta,Anúncios com o erro,Anúncios com a correta,Aprovada? (s/n)
ngoma,engoma,3,122,
rente,frente,3,40,
obrado,sobrado,3,119,
entes,dentes,2,53,
ropria,propria,2,40,
abeca,cabeca,1,24,
amisa,camisa,1,73,
abelo,cabelo,1,24,
ender,vender,1,41,
coutado,acoutado,1,20,



As 10 recusadas mais frequentes:
n = 10


Palavra,Candidata,Anúncios com a palavra,Anúncios com a candidata,Por que foi recusada
alugar,aluga,96,778,frequente
pretas,preta,83,700,frequente
pretos,preto,65,536,frequente
vendem,vende,65,522,frequente
alugao,aluga,51,778,frequente
preco,preto,50,536,frequente
vender,vende,41,522,frequente
venda,vende,40,522,frequente
tudo,todo,33,275,frequente
precisao,precisa,18,162,nao_sistematico


**O que decidir:** quais trocas de OCR a limpeza pode aplicar. O programa troca uma palavra rara por outra muito mais frequente a uma letra de distância, **só quando a rara é a outra sem a 1ª letra** (o erro típico: `luga` por `aluga`). Marque **s** nas trocas que fazem sentido e **n** nas que juntam palavras diferentes. As recusadas são as maiores que **não** foram aplicadas; se alguma deveria ser, diga.

## Parte D. Conferência de uma amostra
**Os rótulos "Claude" são de Claude, não de pessoas.** O grupo preenche as duas últimas colunas.

In [6]:
FIX = RAIZ / "tests" / "fixtures"
def carregar(nome, n=None):
    arq = FIX / nome
    if not arq.exists():
        return []
    itens = json.load(open(arq, encoding="utf-8"))
    return random.Random(42).sample(itens, min(n, len(itens))) if n else itens

def tabela_conferencia(itens):
    return pd.DataFrame([{"id": i["id"], "Período": i.get("periodo"), "Texto (até 300 caracteres)": trecho(i["texto"], 300),
                          "Tipo do código antigo (c4)": i.get("tipo_c4"), "Tipo certo (Claude)": i.get("tipo_certo"),
                          "É anúncio? (Claude)": "sim" if i.get("e_anuncio") else "não",
                          "Tipo certo segundo o grupo": "", "É anúncio de pessoa escravizada? s/n": ""} for i in itens])

print("D1. Os 40 anúncios de conferencia_extra.json")
mostrar(tabela_conferencia(carregar("conferencia_extra.json")), "Arquivo conferencia_extra.json ausente ou vazio.")

D1. Os 40 anúncios de conferencia_extra.json
n = 40


id,Período,Texto (até 300 caracteres),Tipo do código antigo (c4),Tipo certo (Claude),É anúncio? (Claude),Tipo certo segundo o grupo,É anúncio de pessoa escravizada? s/n
7763,1850–1871,"mudar de roupa quando anda fugido, sabe ler e escrever, e talvez se intitule 3 grai quem o apprehender elevará rua dos Benedictinos n. 1; as e fra quem lhe der conto. 2! OM er",fuga,fuga,sim,,
8883,1871–1888,"esarPARECEU da rus do Barão de Itaparipe 14 de Dezembro, uma besta rozilha à pés, tem um joelho enchado; oticias della, paga-se as despezas.",fuga,outro,não,,
13926,antes de 1831,"ma, e sem camisa, quando anda pucha de ham - quarto; quem o apanhar, ou der noticia delle” participando na dita casa receberá alviçaras. ô2 No Domingo de noute 6 do corrente,",fuga,fuga,sim,,
9699,1850–1871,"Desapparecen no dia 15 do corzente, da rua do Peineipo dos Cajueirca D. 159 À, um cachorro pequeno fel- » todo pelo time de Jasmim; quem o á casa seirna ser: grutificado. de",fuga,outro,não,,
13228,1871–1888,"Desoppareceu na quarta-feira, 7 do corrente E rente mez sificaçõe: uma senhora alle gir a assignatura inventor, à do uma filhinha de tres a quatro anno: do onde foi, pede-se aos Srs. proprietari Pharmecia boteis e hospedarias ou pessoas que tiverão conhecimento della o favor de participar na rua da…",fuga,outro,não,,
36026,1850–1871,"fugão, de Los conducta; no campo da Acclamação",fuga,NaN,não,,
34057,1831–1850,"DESAPPARECEU, no dia 3 de fevereiro do correste voso de arsenal do exercito, um aprendiz menor, de nome Francisco Augusto, idade de 14 anvos, natural ce Pernambuco , côr um pouco morena, Lastantes cardos pela cara etem noslto ds cabeça alguma falta de cabello procedida de um tumor que teve he *empo;…",fuga,outro,não,,
19282,1871–1888,"CAPIVARA PRAÇA DO RIO DE JANEIRO FUGIDO iro & C., estabelecidos em Pindamonhangaba, ""O + oa a qu em é Fevereiro do 1854, da fazsada : ça e ego Antonio, natural da Pa- Tm por escripto, ns o, eheio de corpo, À soci Pindamonhangaba, 23 os, bem esperto, de 1885. igeir cicatriz de do pescoço, lado es-",fuga,fuga,sim,,
17437,1871–1888,"individuos desconhecidos na praça. “e faz leilões de moseis de sua casa e de diversas partidas de lonas, ete., de seus estabelecimentos, em nome de terceir Com prejuizos, consideraveis, demonstrando linuidação precipitais, e ainda prejudicando os d Seg hroprio filhinho, fazendo desapparecer bens que…",fuga,outro,não,,
8116,1850–1871,"Fagio no dia 10 do proximo prssad: dio. ta, esorava do Antonio Albino de Souza, me: Torta: peotasta-ab contra que tiver a pe",fuga,fuga,sim,,


In [7]:
print("D2. 60 anúncios de conferencia.json (sorteados, semente 42)")
mostrar(tabela_conferencia(carregar("conferencia.json", 60)), "Arquivo conferencia.json ausente ou vazio.")

D2. 60 anúncios de conferencia.json (sorteados, semente 42)
n = 60


id,Período,Texto (até 300 caracteres),Tipo do código antigo (c4),Tipo certo (Claude),É anúncio? (Claude),Tipo certo segundo o grupo,É anúncio de pessoa escravizada? s/n
13069,1871–1888,"ENDE-SE por 8008 uma preta de meia idade, Dlias “e “uma porta, por 23009 cada um : a; na rua da Alfao- Vperteta cozinheira e lavado trata-se na rua da Candel",venda,venda,sim,,
34068,1831–1850,"FUGIO, da rua do Fogo n. 93, n. dia 20 de muio do corceste anno ums pela de neme Josefa, de nação Moçambique, de 17 a 18 annos, com os pés alguma covsa mal feitos; levou vestido de riscadinho azul lenço encarnado ao pescoço, e tem a falla alguma cousa descansada. Quem della der noticias, ou a levar…",fuga,fuga,sim,,
6120,1831–1850,"DESAPPARECEU, no dia 13 de janeiro do cor« rente anno + hum moleque de nação Congo , “de nome Ludogero, dos que forão aprezados vindos nó bri- 10 gre-escuna Curolina , o qual moleque eu entróguo à Francisco das Chagas Cunha, em 23 de abril dé 1839: levou vestido calças de riscado escuro., camisa de…",fuga,fuga,sim,,
13999,antes de 1831,"45 Domingo 13-do corrente dezappareceu hum muleque de: Nação Benguella, por nome Antomio, de idade pde ll a 12 annos, ainda meio bucal, vestido com camiza de pano de linho, calça de pano azul, einda com algumas sarnas quem delic souber dírija se a rua da Valla no sobrado N. 118, que receberá boas al…",fuga,fuga,sim,,
27116,1850–1871,"va DE set Fui anto-hontesm recolhido de guarda do morro de Panla Mattos, o es- Const rios furancio, como fugido e autor de",fuga,NaN,não,,
23051,1850–1871,"FUGIO ca botica do largo de Santa Rita n. 20, a pon Alfasema ; gratilica-so a quem o apre- 0 SOLO ADO bo MINDELLO. J or nal libers l.",fuga,NaN,não,,
29854,1831–1850,"VENDE-DE um despolpador de café, cujo machinismo é o mais simples possível; wi rua Nova d:s Bento n. 47. — VENDE SE ou aluga se uma preta ama de À com as prendas seguintes : lava, engomma, cose, cogli vestidos € caseisas de jo mem, caseia e é perfeita mucama: aliança se não ter vícios nens molestia,…",venda,venda,sim,,
34219,1831–1850,"ALUGÃO-8E dous pretos bons cozinheiros, e um moleque para pagem ; na rua do Hospício n. 430.",aluguel,aluguel,sim,,
13866,antes de 1831,"“12 Vende-se hum preto sem vicio, ou Marhas, o qual anda ganhango peias ruas, € da todos os dias 240; quem o pertender , dirija. se a rua do Alecrim da Valla para cima N. 55.º j3 Vende se buma preta muito rapariga, de idade 14 a 15 amos, muito sadia, bonita àa, lava, e cosinha-com perfeição, € arrân…",venda,venda,sim,,
6528,1831–1850,"PROCLAMAÇÃO DO NOVO PRESIDENTE, eua t ni hises !— Acaho pelo der 1 ma presidencia e cc ttuto-me, pois, comvyosco, por achar-me à te: hum povo leal, brioso e denodado e por me caber a gloria de ainft huma vez servir ao Erazil e ao throno excelso de o Senhor D. Pedro, renses ! o gento do mal, que não…",fuga,outro,não,,


**O que decidir:** se o nosso código acerta o tipo e se o trecho é anúncio de pessoa escravizada. **Atenção: os rótulos das colunas "Claude" são de Claude (um modelo de linguagem), não de pessoas; podem estar errados.** As duas últimas colunas ficam em branco para o grupo. Onde o grupo discordar, vale a resposta do grupo.